In [ ]:
#Perfect — below is clean, runnable Jupyter Notebook–ready Python code that implements:

#✅ Universal chord parsing
#✅ Letter → number conversion (Nashville system)
#✅ Slash chords
#✅ All chord qualities/extensions preserved
#✅ Chromatic transposition (+ / −)
#✅ Number stays the same when transposing

In [1]:
# Chromatic scale (sharp-based internal representation)
CHROMATIC_SCALE = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

# Enharmonic normalization
ENHARMONIC_MAP = {
    "Db": "C#", "Eb": "D#", "Gb": "F#", "Ab": "G#", "Bb": "A#",
    "Cb": "B", "Fb": "E", "E#": "F", "B#": "C"
}

def normalize_note(note):
    return ENHARMONIC_MAP.get(note, note)


In [2]:
def build_key_map(key):
    key = normalize_note(key)
    start = CHROMATIC_SCALE.index(key)
    
    major_intervals = [0, 2, 4, 5, 7, 9, 11]
    scale = [(CHROMATIC_SCALE[(start + i) % 12], idx + 1)
             for idx, i in enumerate(major_intervals)]
    
    return dict(scale)

#Example usage:
key_map_C = build_key_map("C")
print(key_map_C)  # {'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'A': 6, 'B': 7}

{'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'A': 6, 'B': 7}


In [3]:
#%pip install re
import re

CHORD_REGEX = re.compile(r'^([A-G][b#]?)(.*)$')

def parse_chord(chord):
    if chord == "N.C.":
        return {"root": None, "modifiers": "", "bass": None}

    if "/" in chord:
        chord_part, bass = chord.split("/")
    else:
        chord_part, bass = chord, None

    match = CHORD_REGEX.match(chord_part)
    if not match:
        raise ValueError(f"Invalid chord: {chord}")

    root = normalize_note(match.group(1))
    modifiers = match.group(2)

    bass = normalize_note(bass) if bass else None

    return {
        "root": root,
        "modifiers": modifiers,
        "bass": bass
    }


In [4]:
def note_to_number(note, key_map):
    if note in key_map:
        return str(key_map[note])

    # chromatic offset
    key_notes = list(key_map.keys())
    base = key_notes[0]

    base_idx = CHROMATIC_SCALE.index(base)
    note_idx = CHROMATIC_SCALE.index(note)

    offset = (note_idx - base_idx) % 12
    return f"♯{offset}"


In [5]:
def chord_to_number(chord, key):
    parsed = parse_chord(chord)
    if parsed["root"] is None:
        return "N.C."

    key_map = build_key_map(key)

    root_num = note_to_number(parsed["root"], key_map)
    result = f"{root_num}{parsed['modifiers']}"

    if parsed["bass"]:
        bass_num = note_to_number(parsed["bass"], key_map)
        result += f"/{bass_num}"

    return result


In [6]:
def transpose_note(note, steps):
    idx = CHROMATIC_SCALE.index(note)
    return CHROMATIC_SCALE[(idx + steps) % 12]

def transpose_chord(chord, steps):
    parsed = parse_chord(chord)
    if parsed["root"] is None:
        return "N.C."

    root = transpose_note(parsed["root"], steps)
    bass = transpose_note(parsed["bass"], steps) if parsed["bass"] else None

    result = f"{root}{parsed['modifiers']}"
    if bass:
        result += f"/{bass}"

    return result


In [7]:
def process_chord(key, chord, transpose=0):
    transposed_chord = transpose_chord(chord, transpose)
    number_chord = chord_to_number(chord, key)

    return {
        "original": chord,
        "transposed": transposed_chord,
        "number": number_chord
    }


In [8]:
tests = [
    ("C", "G7♭9/B", 0),
    ("C", "G7♭9/B", 1),
    ("C", "Am7/E", 2),
    ("G", "D/F#", 0),
    ("C", "Cadd9", 1),
]

for key, chord, t in tests:
    print(process_chord(key, chord, t))


{'original': 'G7♭9/B', 'transposed': 'G7♭9/B', 'number': '57♭9/7'}
{'original': 'G7♭9/B', 'transposed': 'G#7♭9/C', 'number': '57♭9/7'}
{'original': 'Am7/E', 'transposed': 'Bm7/F#', 'number': '6m7/3'}
{'original': 'D/F#', 'transposed': 'D/F#', 'number': '5/7'}
{'original': 'Cadd9', 'transposed': 'C#add9', 'number': '1add9'}
